# SketchEffNet - full pipeline

Runs Day 1 to Day 6 in order. It works **on Kaggle** (GPU T4, with the
`quickdraw20` and `imagenetsketch` datasets attached as inputs) and **locally**
(paths come from `sketcheffnet/config.py`, so nothing needs editing).

The recorded results in `results/` came from one run of this pipeline on a Kaggle
Tesla T4 (about 2.5 hours in total). That run, with all of its printed output, is
`notebooks/kaggle_full_run.ipynb`.

**Kaggle:** turn on *Internet* and the *GPU T4* accelerator, add both datasets,
then run all cells. **Locally:** run the setup cell's local branch first
(`python scripts/download_quickdraw.py`).

In [ ]:
import os, sys, subprocess
ON_KAGGLE = os.path.isdir('/kaggle/input')
REPO = 'https://github.com/tanvirul-islam-rifat/sketcheffnet-sketch-recognition.git'

if ON_KAGGLE:
    if not os.path.isdir('/kaggle/working/sketcheffnet-sketch-recognition'):
        subprocess.run(['git', 'clone', '--depth', '1', REPO, '/kaggle/working/sketcheffnet-sketch-recognition'], check=True)
    os.chdir('/kaggle/working/sketcheffnet-sketch-recognition')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'tabulate'], check=True)
else:
    # run the notebook from the repository root (or its notebooks/ folder)
    if os.path.basename(os.getcwd()) == 'notebooks':
        os.chdir('..')
print('cwd:', os.getcwd())

import torch
print('torch', torch.__version__, '| CUDA', torch.cuda.is_available(),
      '| MPS', getattr(torch.backends, 'mps', None) is not None and torch.backends.mps.is_available())

In [ ]:
def run(module, *args):
    """Run one pipeline stage in a fresh interpreter, streaming its output."""
    cmd = [sys.executable, '-m', module, *args]
    print('$', ' '.join(cmd))
    subprocess.run(cmd, check=True)

SMOKE = []          # set to ['--smoke'] for a 2-minute pipeline check

## Check the data

In [ ]:
run('scripts.check_dataset')

## Day 1 - data preparation, EDA, ResNet18 baseline (~15 min on a T4)

In [ ]:
run('experiments.day1_baseline', *SMOKE)

## Day 2 - few-shot, augmentation, transfer (~45 min)

In [ ]:
run('experiments.day2_experiments', *SMOKE)

## Day 3 - analysis tables

In [ ]:
run('experiments.day3_analysis', *SMOKE)

## Day 4 - six CNNs + three classical models (~80 min)

Add `'--with-sketcheffnet'` to also train SketchEffNet under the same protocol.

In [ ]:
run('experiments.day4_model_comparison', *SMOKE)

## Day 5 - SketchEffNet (~37 min)

In [ ]:
run('experiments.day5_sketcheffnet', *SMOKE)

## Day 6 - cross-dataset: ImageNet-Sketch (~6 min)

In [ ]:
run('experiments.day6_cross_dataset_imagenet_sketch', *SMOKE)

## Show the figures

In [ ]:
from IPython.display import Image, display
from sketcheffnet.config import OUTPUT_DIR
root = OUTPUT_DIR / ('smoke' if SMOKE else '') / 'results'
for png in sorted(root.glob('day*/*.png')):
    print(png.relative_to(root)); display(Image(filename=str(png), width=800))